# 演習3. `mutex` で守る ―― 鍵をかけて1スレッドずつ通す

## シーン

演習2で、`std::queue` を保護なしで共有すると壊れることを見ました。
これを直すのが **`std::mutex`（ミューテックス）** です。

`mutex` は**鍵**です。「この区間は一度に1スレッドしか通さない」という札を立てます。

```cpp
std::mutex mtx;                        // 鍵を1つ用意する
...
{
    std::lock_guard<std::mutex> guard(mtx);   // ここで施錠
    ...                                       // 一度に1スレッドしか通れない
}                                             // ここで自動的に解錠
```

**このノートを終えると、共有データを安全に触る書き方が身につきます。**
そして「鍵をかければ安全だが、かけすぎると遅くなる」という肝心のトレードオフも扱います。

## 3-1. 演習2の壊れたキューを直す

`ex02c.cpp` に鍵をかけます。変更点は3行だけです。

In [ ]:
%%writefile ex03a.cpp
#include <iostream>
#include <thread>
#include <queue>
#include <mutex>

std::queue<int> q;
std::mutex mtx;                   // ← ① 鍵を1つ用意する
long taken[2] = {0, 0};

void worker(int id) {
    while (true) {
        {                                              // ← ② ここから鍵をかける区間
            std::lock_guard<std::mutex> guard(mtx);
            if (q.empty()) return;
            q.pop();
        }                                              // ← ③ ここで自動的に鍵が開く
        taken[id]++;
    }
}

int main() {
    for (int i = 0; i < 200000; i++) q.push(i);

    std::thread t1(worker, 0);
    std::thread t2(worker, 1);
    t1.join();
    t2.join();

    std::cout << "スレッド0 = " << taken[0]
              << " / スレッド1 = " << taken[1]
              << " / 合計 = " << taken[0] + taken[1]
              << "   (期待値 200000)\n";
    return 0;
}

In [ ]:
!g++ -std=c++17 -pthread ex03a.cpp -o ex03a
!for i in 1 2 3 4 5; do ./ex03a; done

合計は毎回ぴったり 200,000 になったはずです。
一方で、**スレッド0と1の内訳は毎回違う**ことにも注目してください。
「どちらが何個取るか」は決まっていませんが、「全体で200,000個」は必ず守られます。

### `lock_guard` は「入ったら施錠、出たら解錠」

```cpp
{
    std::lock_guard<std::mutex> guard(mtx);   // この変数が作られた瞬間に施錠
    ...                                       // ここは一度に1スレッドしか通れない
}                                             // 変数が消える＝スコープを抜けた瞬間に解錠
```

自分で「鍵を開ける」と書く必要はありません。`{ }` を抜ければ必ず開きます。
途中で `return` しても、例外が飛んでも、**必ず**開きます。

これは C++問題集の**問8**で出てきた `unique_ptr`（スコープを抜けると自動で解放）と同じ仕組みで、
**RAII** と呼ばれます。C++ でとても重要な考え方です。

`ex03a.cpp` で `if (q.empty()) return;` が鍵の中にあるのに、
ちゃんと解錠されているのはこのおかげです。

## 3-2. 【予測クイズ】鍵をかける区間の広さ

鍵をかければ安全になりますが、**かけすぎると遅くなります**。
鍵の中は一度に1スレッドしか通れない＝**そこだけ並列でなくなる**からです。

次のプログラムは、同じ仕事を「鍵の区間が狭い版」と「広い版」で比べます。
`heavy_work()` は、キューとは関係のない重い計算です。

実行前に予測してください。どちらがどれくらい速いでしょうか。

> コード中の `auto body = [&]() { ... };` はラムダ式です（C++問題集の**問7**を参照）。
> `[&]` は「外の変数をすべて参照で持ち込む」という意味で、
> ここでは `q` や `total` を共有するために使っています。

In [ ]:
%%writefile ex03b.cpp
#include <iostream>
#include <thread>
#include <queue>
#include <mutex>
#include <chrono>
using namespace std::chrono;

std::mutex mtx;

// キューとは関係のない重い計算（1件あたりの処理を模したもの）
long heavy_work(int v) {
    long s = 0;
    for (int i = 0; i < 20000; i++) s += (v + i) % 7;
    return s;
}

long run(bool narrow) {
    std::queue<int> q;
    for (int i = 0; i < 2000; i++) q.push(i);
    long total = 0;

    auto body = [&]() {
        while (true) {
            int v;
            if (narrow) {
                // 【狭い版】キューを触る間だけ施錠し、重い計算は鍵の外でやる
                {
                    std::lock_guard<std::mutex> guard(mtx);
                    if (q.empty()) return;
                    v = q.front(); q.pop();
                }
                long r = heavy_work(v);
                {
                    std::lock_guard<std::mutex> guard(mtx);
                    total += r;
                }
            } else {
                // 【広い版】重い計算まで鍵の中に入れてしまう
                std::lock_guard<std::mutex> guard(mtx);
                if (q.empty()) return;
                v = q.front(); q.pop();
                total += heavy_work(v);
            }
        }
    };

    auto t0 = system_clock::now();
    std::thread t1(body), t2(body);
    t1.join(); t2.join();
    auto t1time = system_clock::now();

    std::cout << (narrow ? "狭い版 : " : "広い版 : ")
              << duration_cast<milliseconds>(t1time - t0).count() << " ms"
              << "   (合計 = " << total << ")\n";
    return total;
}

int main() {
    std::cout << "使えるコア数 = " << std::thread::hardware_concurrency() << "\n";
    long a = run(true);
    long b = run(false);
    std::cout << (a == b ? "結果は一致（どちらも正しい）\n"
                         : "結果が不一致！\n");
    return 0;
}

In [ ]:
!g++ -std=c++17 -pthread ex03b.cpp -o ex03b && ./ex03b

どちらも**答えは正しい**のに、速度が違ったはずです。

広い版は、重い計算をしている間もずっと鍵を握りっぱなしなので、
もう1つのスレッドはただ待っているだけになります。
つまり**2スレッド動かしているのに、実質1スレッド分の速度**しか出ません。

> **鍵は「必要な最小限の区間」にかける。**

C++問題集の問9の解答で、`std::cout` の表示を `{ }` の外に出していたのは、まさにこの理由です。

よくできた「スレッドセーフなキュー」の実装も、鍵の中でやっているのは
`push` / `pop` といった**キューを触る一瞬の操作だけ**です。
取り出したデータをどう処理するかは、必ず鍵の外でやります。

## 3-3. `lock_guard` と `unique_lock`

`mutex` を扱う道具には、`lock_guard` のほかに **`unique_lock`** があります。

```cpp
std::unique_lock<std::mutex> guard(mtx);
```

どちらも「作られたら施錠、スコープを抜けたら解錠」は同じですが、
`unique_lock` は**途中で自分で開け閉めできます**。

In [ ]:
%%writefile ex03c.cpp
#include <iostream>
#include <mutex>

std::mutex mtx;

int main() {
    {
        std::lock_guard<std::mutex> g(mtx);
        std::cout << "lock_guard : 施錠された\n";
        // g.unlock(); と書くことはできない（そういう機能がない）
    }
    std::cout << "lock_guard : スコープを抜けて解錠\n\n";

    {
        std::unique_lock<std::mutex> g(mtx);
        std::cout << "unique_lock: 施錠された   (owns_lock = " << g.owns_lock() << ")\n";
        g.unlock();                                  // ← 途中で開けられる
        std::cout << "unique_lock: 途中で解錠   (owns_lock = " << g.owns_lock() << ")\n";
        g.lock();                                    // ← また閉められる
        std::cout << "unique_lock: また施錠     (owns_lock = " << g.owns_lock() << ")\n";
    }
    std::cout << "unique_lock: スコープを抜けて解錠\n";
    return 0;
}

In [ ]:
!g++ -std=c++17 -pthread ex03c.cpp -o ex03c && ./ex03c

`lock_guard` のほうが単純で安全なので、**ふつうは `lock_guard` を使います**。

では `unique_lock` はいつ使うのか。
「途中で開け閉めできる」という、まさにこの機能が必要になる場面があります。

- 条件（キューが空でない、など）が満たされるまで**待つ**あいだ、鍵を握ったままでは
  他のスレッドがキューに入れられず、永遠に条件が満たされません
- そこで「**待っているあいだは鍵を開けておき、起こされたらまた閉める**」必要があります
- これができるのが `unique_lock` だけなので、`condition_variable` は `unique_lock` を要求します

その `condition_variable` は**演習4**で扱います。
いまは「`unique_lock` は途中で開け閉めできる版の `lock_guard`」と覚えておけば十分です。

## 発展課題

1. `ex03a.cpp` の `{ }` を外して、`lock_guard` を `while` ループの先頭に置いたらどうなるでしょうか。
   ```cpp
   while (true) {
       std::lock_guard<std::mutex> guard(mtx);
       if (q.empty()) return;
       q.pop();
       taken[id]++;
   }
   ```
   このコードは**正しく動くか**、**速いか**、それぞれ考えてください。

2. `ex03b.cpp` の `heavy_work` の計算量（`20000` の部分）を10倍・1/10にすると、
   狭い版と広い版の差はどう変わると思いますか。予測してから確かめてください。

3. 鍵が2つある場面を考えます。スレッドAが「鍵1 → 鍵2」の順にロックし、
   スレッドBが「鍵2 → 鍵1」の順にロックしたら何が起きるでしょうか。
   （これは **デッドロック** と呼ばれる現象です。コードは書かず、図を描いて考えてみてください）

4. 「キューにデータを入れたので、待っている人に知らせる」という処理を、
   次のように**鍵を握ったまま**書いたとします。
   ```cpp
   {
       std::lock_guard<std::mutex> g(mtx);
       queue.push(value);
       notify();          // ← まだ施錠されている
   }
   ```
   「通知は鍵を開けてから出すべきだ」という意見があります。なぜでしょうか。
   起こされた側が何をしなければならないかを考えてみてください。
   （演習4を終えてから戻ってくると分かりやすくなります）